In [1]:
import pandas as pd
import os

In [ ]:
INPUT_CSV = "target_cabang.csv"
OUTPUT_DIR = "output_sql"
TAHUN = 2026
DB_NAME = "u194014241_kpi_db" 

# True = file per area 
# False = 1 file gabungan 
SPLIT_PER_AREA = True


In [ ]:
# MAPPING: CSV CODE -> DB CODE

CSV_CODE_TO_DB = {
   "CM" : "CM", 
   "CIR" : "CIR", 
   "FBI" : "FBI", 
   "ACE" : "ACE_GROWTH", 
   "DPK" : "DPK", 
   "TAB" : "TAB_GROWTH", 
   "GIRO" : "GIRO_GROWTH", 
   "DEPO" : "DEPOSITO_GROWTH", 
   "USAK" : "USAK_GROWTH", 
   "TABEMAS(GR)" : "TABEMAS", 
   "PBY" : "PBY", 
   "KONSUMER" : "KONSUMER_GROWTH", 
   "SME" : "SME_GROWTH", 
   "Mikro" : "MIKRO_GROWTH", 
   "GPB" : "GPB_GROWTH", 
   "NPF" : "NPF", 
   "KOL2" : "KOL_2", 
   "CBC" : "CBC", 
   "CBR" : "CBR", 
   "PHE" : "PHE", 
   "NoaPayroll" : "NOA_PAYROLL", 
   "NoaHaji" : "NOA_HAJI", 
   "NoaEmas" : "NOA_EMAS", 
   "ProdSales" : "PROD_SALES", 
   "GRIYA" : "GRIYA_GROWTH", 
   "OTO" : "OTO_GROWTH", 
   "MITRAGUNA" : "MITRAGUNA_GROWTH", 
   "PENSIUN" : "PENSIUN_GROWTH", 
   "OprIndex" : "OPRINDEX", 
   "HCIndex" : "HCINDEX", 
}


In [ ]:
# MAPPING: DB CODE -> VARIABLE ID

VAR_CODE_TO_ID = { 
    "CM" : 1, 
    "CIR" : 2, 
    "FBI" : 3, 
    "ACE_GROWTH" : 4, 
    "DPK" : 5, 
    "TAB_GROWTH" : 6, 
    "GIRO_GROWTH" : 7, 
    "DEPOSITO_GROWTH" : 8, 
    "USAK_GROWTH" : 9, 
    "TABEMAS" : 10, 
    "PBY" : 11, 
    "KONSUMER_GROWTH" : 12, 
    "SME_GROWTH" : 13, 
    "MIKRO_GROWTH" : 14, 
    "GPB_GROWTH" : 15, 
    "NPF" : 16, 
    "KOL_2" : 17, 
    "CBC" : 18, 
    "CBR" : 19, 
    "PHE" : 20, 
    "NOA_PAYROLL" : 21, 
    "NOA_HAJI" : 22, 
    "NOA_EMAS" : 23, 
    "PROD_SALES" : 24, 
    "GRIYA_GROWTH" : 25, 
    "OTO_GROWTH" : 26, 
    "MITRAGUNA_GROWTH" : 27, 
    "PENSIUN_GROWTH" : 28, 
    "OPRINDEX" : 29, 
    "HCINDEX" : 30, 
}

In [ ]:
# MAPPING CABANG

BRANCH_NAME_TO_ID = {     
    # ── Area Banjarmasin (branch_id 1–26)
    "KC BANJARMASIN LAMBUNG MANGKURAT" : 1,
    "KCP PALANGKARAYA 1" : 2, 
    "KC MARTAPURA" : 3, 
    "KCP MARTAPURA" : 3, # nama lama di CSV 
    "KCP BANJARMASIN GATOT SUBROTO" : 4, 
    "KCP BATULICIN" : 5, 
    "KCP BATULICIN JHONLIN" : 5, # nama lama di CSV 
    "KCP BARABAI" : 6, 
    "KCP BANJARMASIN SENTRA ANTASARI" : 7, 
    "KCP PELAIHARI" : 8, 
    "KCP KOTABARU" : 9, 
    "KC PANGKALAN BUN" : 10, 
    "KC TANJUNG" : 11, 
    "KC SAMPIT" : 12, 
    "KCP KAPUAS" : 13, 
    "KCP AMUNTAI" : 14, 
    "KCP MUARA TEWEH" : 15, 
    "KCP BANJAR KERTAK HANYAR" : 16, 
    "KCP TAPIN" : 17, 
    "KCP BANJARMASIN KAYU TANGI" : 18, 
    "KCP BANJARMASIN A YANI 1" : 19, 
    "KC PALANGKARAYA DIPONEGORO" : 20, 
    "KC BANJARBARU" : 21, 
    "KCP SUNGAI DANAU" : 22, 
    "KCP MARTAPURA PASAR INTAN" : 23, 
    "KCP BANJARBARU LANDASAN ULIN" : 24, 
    "KCP BANJARBARU A YANI" : 24, # nama lama di CSV = cabang yang sama 
    "KCP BARITO KUALA" : 25, 
    "KCP KANDANGAN" : 26,
    # ── Area Balikpapan (branch_id 27–51)
    "KC BALIKPAPAN SUDIRMAN" : 27, 
    "KC BALIKPAPAN SUDIRMAN 1" : 27, # nama lama di CSV 
    "KC SAMARINDA ANTASARI" : 28, 
    "KC KUTAI KARTANEGARA" : 29, 
    "KC BONTANG" : 30, 
    "KC TARAKAN" : 31, 
    "KCP BALIKPAPAN SEPINGGAN" : 32, 
    "KCP BALIKPAPAN BARU" : 33, 
    "KCP BALIKPAPAN BARU 1" : 33, # nama lama di CSV 
    "KCP SAMARINDA PAHLAWAN" : 34, 
    "KCP SANGATTA" : 35, 
    "KCP SANGATTA 1" : 35, # nama lama di CSV 
    "KCP BALIKPAPAN KEBUN SAYUR" : 36, 
    "KCP SAMARINDA DI PANJAITAN" : 37, 
    "KCP KUTAI SENDAWAR" : 38, 
    "KCP PENAJAM" : 39, 
    "KCP SAMARINDA SUDIRMAN" : 40, 
    "KCP KUTAI TENGGARONG SEBERANG" : 41, 
    "KCP NUNUKAN" : 42, 
    "KCP NUSANTARA" : 43, 
    "KCP BALIKPAPAN SOEKARNO HATTA" : 44, 
    "KCP SAMARINDA BUNG TOMO" : 45, 
    "KCP PASER TANAH GROGOT" : 46, 
    "KCP BERAU" : 47, 
    "KCP BALIKPAPAN SUDIRMAN 1" : 48, 
    "KCP SAMARINDA JUANDA" : 49, 
    "KCP BULUNGAN" : 50, 
    "KCP BONTANG BARU" : 51, 
    "KCP BONTANG" : 51, # nama lama di CSV 
    # ── Area Pontianak (branch_id 52–68)     
    "KC PONTIANAK ABDURRACHMAN" : 52, 
    "KC KETAPANG" : 53, 
    "KCP SINTANG LINTAS MELAWI" : 54, 
    "KCP NANGA PINOH" : 55, 
    "KCP PONTIANAK DIPONEGORO" : 56, 
    "KC SAMBAS" : 57, 
    "KCP PONTIANAK A YANI 1" : 58, 
    "KCP PONTIANAK SUNGAI JAWI" : 59, 
    "KCP KETAPANG MANIS MATA" : 60, 
    "KCP SANGGAU SUDIRMAN" : 61, 
    "KC SINGKAWANG" : 62, 
    "KCP KUBU RAYA" : 63, 
    "KCP PONTIANAK SIANTAN" : 64, 
    "KCP SANGGAU A YANI" : 65, 
    "KCP MEMPAWAH" : 66, 
    "KCP PONTIANAK A YANI 3" : 67, 
    "KCP PUTUSSIBAU" : 68,
}

In [ ]:
# ==================================================================== # AREA # ==================================================================== 
AREA_BRANCH_IDS = { 
    "banjarmasin" : list(range(1, 27)), 
    "balikpapan" : list(range(27, 52)), 
    "pontianak" : list(range(52, 69)), 
} 

# ==================================================================== # MONTH COLUMNS # ==================================================================== 
MONTH_COLUMNS = {
    f"Jan-{str(TAHUN)[2:]}" : f"{TAHUN}-01-01", 
    f"Feb-{str(TAHUN)[2:]}" : f"{TAHUN}-02-01", 
    f"Mar-{str(TAHUN)[2:]}" : f"{TAHUN}-03-01", 
    f"Apr-{str(TAHUN)[2:]}" : f"{TAHUN}-04-01", 
    f"May-{str(TAHUN)[2:]}" : f"{TAHUN}-05-01", 
    f"Jun-{str(TAHUN)[2:]}" : f"{TAHUN}-06-01", 
    f"Jul-{str(TAHUN)[2:]}" : f"{TAHUN}-07-01", 
    f"Aug-{str(TAHUN)[2:]}" : f"{TAHUN}-08-01", 
    f"Sep-{str(TAHUN)[2:]}" : f"{TAHUN}-09-01", 
    f"Oct-{str(TAHUN)[2:]}" : f"{TAHUN}-10-01", 
    f"Nov-{str(TAHUN)[2:]}" : f"{TAHUN}-11-01", 
    f"Dec-{str(TAHUN)[2:]}" : f"{TAHUN}-12-01", 
}


In [ ]:
# ====================================================================
# FUNCTIONS
# ====================================================================

def clean_number(val):
    if pd.isna(val):
        return None
    try:
        return float(str(val).strip().replace(",", ""))
    except ValueError:
        return None


def build_rows(df):
    rows = {}
    errors = []

    for _, row in df.iterrows():
        raw_name = str(row["Nama Cabang"]).strip()
        csv_code = str(row["Kode1"]).strip()

        branch_id = BRANCH_NAME_TO_ID.get(raw_name)
        db_code   = CSV_CODE_TO_DB.get(csv_code)
        var_id    = VAR_CODE_TO_ID.get(db_code) if db_code else None

        if branch_id is None:
            errors.append(f'[SKIP] Cabang tidak ditemukan: "{raw_name}"')
            continue

        if var_id is None:
            errors.append(f'[SKIP] Variable tidak ditemukan: "{csv_code}"')
            continue

        for col, periode in MONTH_COLUMNS.items():

            if col not in df.columns:
                continue

            num = clean_number(row[col])

            val = "NULL" if num is None else repr(num)

            key = (branch_id, var_id, periode)

            if key not in rows or val != "NULL":
                rows[key] = val

    return list(rows.items()), errors


def write_sql(rows, filepath, label):

    filled = sum(1 for _, v in rows if v != "NULL")
    nulls  = sum(1 for _, v in rows if v == "NULL")

    with open(filepath, "w", encoding="utf-8") as f:

        f.write(f"/* kpi_targets - {label} - Jan sd Des {TAHUN} */\\n")
        f.write(f"/* Total: {len(rows)} baris | Filled: {filled} | NULL: {nulls} */\\n\\n")

        f.write(f"USE {DB_NAME};\\n\\n")

        f.write(
            "INSERT IGNORE INTO kpi_targets "
            "(branch_id, variable_id, periode, target_value) VALUES\\n"
        )

        for i, ((bid, vid, periode), val) in enumerate(rows):

            sep = ";" if i == len(rows) - 1 else ","

            f.write(
                f"  ({bid}, {vid}, '{periode}', {val}){sep}\\n"
            )

    size_kb = os.path.getsize(filepath) / 1024

    print(
        f"OK {os.path.basename(filepath)} "
        f"| {len(rows):,} rows | {size_kb:.0f} KB"
    )

In [ ]:
# ==================================================================== # MAIN PROCESS # ==================================================================== 
print("=" * 60) 
print("KPI TARGET IMPORTER") 
print("=" * 60) 

if not os.path.exists(INPUT_CSV): 
    print(f"[ERROR] File tidak ditemukan: {INPUT_CSV}") 
    
else: 
    df = pd.read_csv(INPUT_CSV, sep=None, engine="python") 

    print(f"File CSV : {INPUT_CSV}") 
    print(f"Total baris : {len(df):,}") 
    print(f"Kolom : {df.columns.tolist()}") 

    all_rows, errors = build_rows(df) 

    print("\\nHasil Mapping") 
    print(f"Total rows : {len(all_rows):,}") 
    
    if errors: 
        print("\\nPeringatan:") 
        for e in sorted(set(errors)): 
            print(e) 
            
    os.makedirs(OUTPUT_DIR, exist_ok=True) 
    
    if SPLIT_PER_AREA: 
        for area_name, branch_ids in AREA_BRANCH_IDS.items(): 
            area_rows = [ 
                        ((bid, vid, per), val) 
                        for (bid, vid, per), val in all_rows 
                        if bid in branch_ids 
                        ] 
            
            out_path = os.path.join( OUTPUT_DIR, f"kpi_targets_{TAHUN}_{area_name}.sql" ) 
            write_sql( area_rows, out_path, f"Area {area_name.upper()}" ) 

    else: 
        out_path = os.path.join( OUTPUT_DIR, f"kpi_targets_{TAHUN}_all.sql" ) 
        write_sql(all_rows, out_path, "Semua Area") 
        print("\\nSelesai 🚀")
